The TIMS data used for these indicators are downloaded here at this tool:  https://tims.berkeley.edu/tools/safetypm

Choose the "SWITRS" option for the *Facility data source*.

The raw data needs to be downloaded at every level within the SACOG region and stored here:  https://sacog.sharepoint.com/sites/RegionalMonitoringandReporting/Shared%20Documents/Forms/AllItems.aspx?id=%2Fsites%2FRegionalMonitoringandReporting%2FShared%20Documents%2FData%2FSafe%20Equitable%20Resilient%20Infrastructure%2FSafety%2FTIMS&viewid=8fba2a3f%2Dea8a%2D4663%2Da31f%2Dc1760cb6922d


For example, in the "Fatalities" folder, there is a folder for counties, jurisdictions, MPO, and statewide.  Step through the tool and download the data one geography at a time.  For the "Counties" folder, us the *County* drop-down option and set it to "El Dorado".  Leave the *City* drop-down option set to "Show Countywide".  There should be two 6 charts total when you scroll down the webpage (2 for fatalities, 2 for serious injuries, and 2 for non-motorized), but for the "Fatalities" folder, only download a *csv* file from the first two charts.  Rename them to match the naming convention that is already in the "Counties" folder and copy/paste them to replace the old files.

Repeat this process for all counties and all jurisdictions.  Then pull the data for the SACOG MPO and CA statewide.

Then, repeat this process for the "Serious Injuries" and "Non-Motorized" folders.


Once all the data is downloaded, rerun this script (make sure that you renamed the files properly to import correctly here and run smoothly).  Check the plots to make sure that the data looks appropriate.  If the outputs look good, then uncomment all blocks of code that export the results.  Then rerun the script and check the outputs on SharePoint.

In [28]:


# TIMS does not have an API available.  Data were manually downloaded from here https://tims.berkeley.edu/ and stored on SharePoint


import functools as ft
import os
import re
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

sys.path.append(str(Path.cwd().parent.parent / 'config'))
import functions as func

pd.set_option('display.max_columns', None)


def calc_statewide(categories, years_to_import):


    dt_cat4 = {}

    for cat in categories:
        path_cat = PATH_TIMS / cat / 'Statewide'

        list_files = []
        for file in os.listdir(path_cat):
            path_mpo = os.path.join(path_cat, file)
            list_files.append(path_mpo)        
        dt_cat4[cat] = list_files

    print(dt_cat4)


    print('Organizing raw TIMS data...')


    list_df_final = []
    for cat in dt_cat4.keys():
        print(cat)

        list_df_total = []
        list_df_mvmt  = []
        list_df_non1  = []
        list_df_non2  = []
        
        for path_csv in dt_cat4[cat]:
            df_temp = pd.read_csv(path_csv)
            file = re.sub(r'.*\\', '', path_csv)
            
            df_temp = df_temp.iloc[:,0:3]

            metric = re.sub('-', ' ', re.sub(r'-[^-]*$', '', file))
            metric = metric.capitalize()
            if '100 mvmt' in metric:
                metric = re.sub(' 100 mvmt', '_100 mvmt', metric)
            df_temp.columns = ['Year', metric, metric + '_5 Year Rolling Average']

            df_temp = df_temp[df_temp['Year'].isin(years_to_import)]

            state = re.sub('.csv', '', re.sub(r'.*-', '', file))
            
            df_temp['State'] = state

            df_temp = df_temp.set_index(['State', 'Year']).reset_index()

            if cat == 'Non-Motorized':
                if 'serious' in file:
                    list_df_non2.append(df_temp)
                else:
                    list_df_non1.append(df_temp)
            else:
                if 'mvm' in file:
                    list_df_mvmt.append(df_temp)
                else:
                    list_df_total.append(df_temp)
        if cat == 'Non-Motorized':                
            df_non1  = pd.concat(list_df_non1 )
            df_non2  = pd.concat(list_df_non2)
            list_df_cat = [df_non1, df_non2]
        else:                
            df_mvmt  = pd.concat(list_df_mvmt )
            df_total = pd.concat(list_df_total)
            list_df_cat = [df_total, df_mvmt]

        df_cat = ft.reduce(lambda left, right: pd.merge(left, right, on = ['State', 'Year']), list_df_cat)
        list_df_final.append(df_cat)

    df_tims4 = ft.reduce(lambda left, right: pd.merge(left, right, on = ['State', 'Year']), list_df_final)
    df_tims4 = df_tims4.sort_values(['State', 'Year'], ascending = [True, True])

    print('\nFinished !!\n')

    return df_tims4



PATH_SERVER = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")
PATH_OUT  = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' / 'Data' / 'Safe Equitable Resilient Infrastructure' / 'Safety'
PATH_TIMS = PATH_OUT / 'TIMS'
FILE_ABOUT = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' / 'Process Revamp' / 'Task 6. Process Map' / 'About Indicators.xlsx'



EXPORT=True


In [29]:

year_start = 2012
year_end = 2025
years_to_import = range(year_start, year_end+1)

categories = [cat for cat in os.listdir(PATH_TIMS) if '.xlsx' not in cat]
categories.remove('Bicyclists vs Pedestrians')

print(categories, years_to_import)


['Fatalities', 'Non-Motorized', 'Serious Injuries'] range(2012, 2026)


In [30]:


# Import data statewide
df_tims4 = calc_statewide(categories, years_to_import)
display(df_tims4.head())

if EXPORT:
    with pd.ExcelWriter(PATH_TIMS / 'TIMS SWITRS Data Statewide.xlsx', engine='xlsxwriter') as writer:
        df_tims4.to_excel(writer, index=False, sheet_name='Statewide')



{'Fatalities': ['C:\\Users\\jfontes\\Sacramento Area Council of Governments\\Regional Monitoring and Reporting - Documents\\Data\\Safe Equitable Resilient Infrastructure\\Safety\\TIMS\\Fatalities\\Statewide\\fatalities-100-mvmt-CA.csv', 'C:\\Users\\jfontes\\Sacramento Area Council of Governments\\Regional Monitoring and Reporting - Documents\\Data\\Safe Equitable Resilient Infrastructure\\Safety\\TIMS\\Fatalities\\Statewide\\fatalities-CA.csv'], 'Non-Motorized': ['C:\\Users\\jfontes\\Sacramento Area Council of Governments\\Regional Monitoring and Reporting - Documents\\Data\\Safe Equitable Resilient Infrastructure\\Safety\\TIMS\\Non-Motorized\\Statewide\\non-motorized-fatalities-CA.csv', 'C:\\Users\\jfontes\\Sacramento Area Council of Governments\\Regional Monitoring and Reporting - Documents\\Data\\Safe Equitable Resilient Infrastructure\\Safety\\TIMS\\Non-Motorized\\Statewide\\non-motorized-serious-in-CA.csv'], 'Serious Injuries': ['C:\\Users\\jfontes\\Sacramento Area Council of Gove

,State,Year,Fatalities,Fatalities_5 Year Rolling Average,Fatalities_100 mvmt,Fatalities_100 mvmt_5 Year Rolling Average,Non motorized fatalities,Non motorized fatalities_5 Year Rolling Average,Non motorized serious in,Non motorized serious in_5 Year Rolling Average,Serious injuries,Serious injuries_5 Year Rolling Average,Serious injuries_100 mvmt,Serious injuries_100 mvmt_5 Year Rolling Average
0,CA,2015,3435.0,NaN,1.011,NaN,1008.0,NaN,2798.0,NaN,11950.0,NaN,3.516,NaN
1,CA,2016,3854.0,NaN,1.124,NaN,1129.0,NaN,3016.0,NaN,13258.0,NaN,3.867,NaN
2,CA,2017,3904.0,NaN,1.134,NaN,1125.0,NaN,3174.0,NaN,14201.0,NaN,4.124,NaN
3,CA,2018,3804.0,NaN,1.096,NaN,1173.0,NaN,3397.0,NaN,16158.0,NaN,4.656,NaN
4,CA,2019,3737.0,3746.8,1.064,1.086,1195.0,1126.0,3500.0,3177.0,16443.0,14402.0,4.682,4.169


In [ ]:

# Safety_1 Collision Rates
df_safety1_4 = df_tims4[['State', 'Year', 'Fatalities'              , 'Fatalities_5 Year Rolling Average'
                                        , 'Fatalities_100 mvmt'     , 'Fatalities_100 mvmt_5 Year Rolling Average'
                                        , 'Serious injuries'        , 'Serious injuries_5 Year Rolling Average'
                                        , 'Serious injuries_100 mvmt', 'Serious injuries_100 mvmt_5 Year Rolling Average']]

# Safety_2 Non-Motorized
df_safety2_4 = df_tims4[['State', 'Year', 'Non motorized fatalities', 'Non motorized fatalities_5 Year Rolling Average'
                                        , 'Non motorized serious in', 'Non motorized serious in_5 Year Rolling Average']]


if EXPORT:

    # Update overall about documentations workbook
    sample_type = 'TIMS'
    indicators = ['Safety_1', 'Safety_2']
    params = {
        'sample': sample_type
        , 'start_year': year_start
        , 'end_year': year_end
        , 'estimate': None
        , 'moe_thresh': None
    }

    categories = ['Collision Rates', 'Non-Motorized']
    paths = [PATH_OUT, PATH_SERVER]

    for path in paths:
        for category in categories:
            if category == 'Collision Rates':
                params['indicator'] = 'Safety_1'
                dt_geo = {'Statewide': df_safety1_4}
            if category == 'Non-Motorized':
                params['indicator'] = 'Safety_2'
                dt_geo = {'Statewide': df_safety2_4}
            for geography, df in dt_geo.items():
                params['geo']=geography
                df_about = func.write_about(params)
                if path == PATH_OUT:
                    file_out = path / f'{params['indicator']} {category}' / f'{params['indicator']} {params['geo']} {params['sample']}_TEST.xlsx'
                if path == PATH_SERVER:
                    file_out = path / f'{params['indicator']} {params['geo']} {params['sample']}_TEST.xlsx'
                print('Excel files exported here: ' + str(file_out))
                with pd.ExcelWriter(file_out, engine='xlsxwriter') as writer:
                    df_about.to_excel(writer, index=False, sheet_name='About'  , header=False)
                    df      .to_excel(writer, index=False, sheet_name=geography              )

                


KeyError: "['Serious Injuries', 'Serious Injuries_5 Year Rolling Average', 'Serious Injuries_100 mvmt', 'Serious Injuries_100 mvmt_5 Year Rolling Average'] not in index"